# RAG-Based Anomaly Explainer

An end-to-end LLM pipeline that auto-explains time-series anomalies using Retrieval-Augmented Generation (RAG).  
Detected anomalies are matched against a knowledge base of past incidents and root causes, then explained in plain language using an LLM.

**Stack:** Python · LangChain · ChromaDB · HuggingFace Transformers · OpenAI API · AWS SageMaker (deployment)

---
## Pipeline Overview
```
Time-Series Data
      ↓
Anomaly Detection (Z-score / IQR)
      ↓
Feature Extraction (context window, severity, trend)
      ↓
ChromaDB Vector Store (past incident knowledge base)
      ↓
RAG Retrieval → LLM (GPT / open-source)
      ↓
Natural Language Explanation + Recommended Action
```

## 1. Install Dependencies

In [ ]:
# !pip install langchain langchain-community langchain-openai chromadb
# !pip install sentence-transformers openai pandas numpy matplotlib scikit-learn

## 2. Imports & Config

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# Set your OpenAI key or use a local model
# os.environ["OPENAI_API_KEY"] = "your-key-here"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DIR  = "./chroma_store"
print("Imports OK")

## 3. Synthetic Time-Series Data with Injected Anomalies

In [ ]:
def generate_timeseries(n=500, seed=42):
    """Generate synthetic audience measurement data with anomalies."""
    np.random.seed(seed)
    dates = [datetime(2024, 1, 1) + timedelta(hours=i) for i in range(n)]
    # Base signal: daily seasonality + trend
    t = np.arange(n)
    signal = 1000 + 50 * np.sin(2 * np.pi * t / 24) + 0.1 * t + np.random.normal(0, 10, n)
    # Inject anomalies
    anomaly_indices = [120, 240, 380]
    signal[120] += 300   # spike — data pipeline flush
    signal[240] -= 250   # drop  — feed outage
    signal[380] += 400   # spike — bot traffic
    df = pd.DataFrame({"timestamp": dates, "value": signal})
    return df, anomaly_indices

df, true_anomalies = generate_timeseries()

plt.figure(figsize=(14, 4))
plt.plot(df["timestamp"], df["value"], linewidth=0.8, label="Signal")
for idx in true_anomalies:
    plt.axvline(df["timestamp"][idx], color="red", linestyle="--", alpha=0.7, label="Anomaly" if idx == true_anomalies[0] else "")
plt.title("Synthetic Audience Metric — Injected Anomalies")
plt.legend()
plt.tight_layout()
plt.show()
print(df.head())

## 4. Anomaly Detection (Z-Score + IQR)

In [ ]:
def detect_anomalies(df, z_thresh=3.0, window=24):
    """Rolling Z-score anomaly detection."""
    df = df.copy()
    df["rolling_mean"] = df["value"].rolling(window, center=True, min_periods=1).mean()
    df["rolling_std"]  = df["value"].rolling(window, center=True, min_periods=1).std().fillna(1)
    df["z_score"] = (df["value"] - df["rolling_mean"]) / df["rolling_std"]
    df["is_anomaly"] = df["z_score"].abs() > z_thresh
    df["direction"]  = np.where(df["z_score"] > 0, "spike", "drop")
    df["severity"]   = df["z_score"].abs().round(2)
    return df

df_detected = detect_anomalies(df)
anomalies = df_detected[df_detected["is_anomaly"]].reset_index(drop=True)
print(f"Detected {len(anomalies)} anomalies:")
print(anomalies[["timestamp", "value", "z_score", "direction", "severity"]])

## 5. Knowledge Base — Past Incidents

In [ ]:
INCIDENT_KB = [
    {
        "id": "INC-001",
        "type": "spike",
        "description": "Large positive spike in audience count.",
        "root_cause": "Data pipeline flush — delayed records from upstream system were batched and delivered at once.",
        "resolution": "Implement incremental ingestion with deduplication. Monitor upstream lag metrics.",
        "tags": "spike pipeline flush batch ingestion"
    },
    {
        "id": "INC-002",
        "type": "drop",
        "description": "Sudden drop in audience metric below expected range.",
        "root_cause": "Feed outage — data provider API went offline, causing a gap in incoming records.",
        "resolution": "Set up provider health checks and fallback imputation for short outages.",
        "tags": "drop outage feed API data loss"
    },
    {
        "id": "INC-003",
        "type": "spike",
        "description": "Unusually high spike in traffic or audience count.",
        "root_cause": "Bot or non-human traffic detected — automated crawlers inflated the audience metric.",
        "resolution": "Apply bot-filtering rules and validate against panel-based ground truth.",
        "tags": "spike bot traffic inflated non-human"
    },
    {
        "id": "INC-004",
        "type": "spike",
        "description": "Moderate spike tied to scheduled event.",
        "root_cause": "Planned marketing campaign or live event drove genuine audience surge.",
        "resolution": "Cross-reference with campaign calendar before flagging. Consider expected-event whitelisting.",
        "tags": "spike campaign event marketing genuine"
    },
    {
        "id": "INC-005",
        "type": "drop",
        "description": "Gradual decline in metric values over several hours.",
        "root_cause": "Schema change in upstream data caused silent parse failures and data loss.",
        "resolution": "Add schema validation checks at ingestion layer with alerting on parse error rates.",
        "tags": "drop schema parse failure gradual decline"
    },
]

# Convert to LangChain Documents
kb_docs = [
    Document(
        page_content=f"[{inc['id']}] {inc['description']} Root cause: {inc['root_cause']} Resolution: {inc['resolution']}",
        metadata={"id": inc["id"], "type": inc["type"], "tags": inc["tags"]}
    )
    for inc in INCIDENT_KB
]
print(f"Knowledge base: {len(kb_docs)} incident records loaded.")

## 6. Build ChromaDB Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

vectorstore = Chroma.from_documents(
    documents=kb_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("ChromaDB vector store built and persisted.")

## 7. RAG Prompt Template

In [ ]:
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert data quality analyst for an audience measurement platform.
Use the past incident records below to explain the anomaly and suggest a resolution.

Past Incidents (retrieved context):
{context}

Current Anomaly:
{question}

Provide:
1. Most likely root cause (1-2 sentences)
2. Confidence level: High / Medium / Low
3. Recommended action (1-2 sentences)
4. Similar past incident ID if applicable
"""
)

## 8. RAG Chain & Explanation Generator

In [ ]:
def build_anomaly_query(row):
    """Convert a detected anomaly row into a natural language query."""
    return (
        f"A {row['direction']} anomaly was detected at {row['timestamp']} "
        f"with value {row['value']:.1f} (Z-score: {row['z_score']:.2f}, severity: {row['severity']}). "
        f"The rolling mean at that time was {row['rolling_mean']:.1f}. "
        f"What is the likely root cause and what action should be taken?"
    )

def explain_anomaly(row, retriever, prompt_template):
    """Retrieve similar incidents and generate an explanation."""
    query = build_anomaly_query(row)

    # Retrieve relevant past incidents
    docs = retriever.get_relevant_documents(query)
    context = "\n\n".join([d.page_content for d in docs])

    # Format prompt (in production, send to LLM — mocked here)
    filled_prompt = prompt_template.format(context=context, question=query)

    # --- MOCK LLM RESPONSE (replace with real LLM call) ---
    # llm = ChatOpenAI(model="gpt-4o", temperature=0)
    # response = llm.invoke(filled_prompt).content
    mock_responses = {
        "spike": (
            "1. Root cause: Likely a data pipeline flush or bot traffic inflating the metric.\n"
            "2. Confidence: Medium\n"
            "3. Action: Check upstream pipeline lag and validate against panel ground truth.\n"
            "4. Similar incident: INC-001 or INC-003"
        ),
        "drop": (
            "1. Root cause: Likely a feed outage or schema parse failure causing data loss.\n"
            "2. Confidence: High\n"
            "3. Action: Check provider API health and inspect ingestion parse error logs.\n"
            "4. Similar incident: INC-002 or INC-005"
        )
    }
    response = mock_responses.get(row["direction"], "Unable to determine root cause.")
    # --------------------------------------------------------

    return {"query": query, "retrieved_context": context, "explanation": response}

print("Explanation function ready.")

## 9. Run Explanations on All Detected Anomalies

In [ ]:
results = []
for _, row in anomalies.iterrows():
    result = explain_anomaly(row, retriever, RAG_PROMPT)
    results.append(result)

for i, r in enumerate(results):
    print(f"\n{'='*60}")
    print(f"ANOMALY {i+1}")
    print(f"Query:\n  {r['query']}")
    print(f"\nExplanation:\n{r['explanation']}")
    print(f"{'='*60}")

## 10. Export Results

In [ ]:
output = []
for i, (_, row) in enumerate(anomalies.iterrows()):
    output.append({
        "timestamp": str(row["timestamp"]),
        "value": round(row["value"], 2),
        "z_score": round(row["z_score"], 2),
        "direction": row["direction"],
        "severity": row["severity"],
        "explanation": results[i]["explanation"]
    })

with open("anomaly_explanations.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to anomaly_explanations.json")
print(json.dumps(output[0], indent=2))

## 11. AWS SageMaker Deployment (Reference)

To deploy this as a REST endpoint on SageMaker:

```python
import boto3, sagemaker
from sagemaker.huggingface import HuggingFaceModel

role = sagemaker.get_execution_role()

hub = {
    'HF_MODEL_ID': 'sentence-transformers/all-MiniLM-L6-v2',
    'HF_TASK': 'feature-extraction'
}

model = HuggingFaceModel(
    env=hub,
    role=role,
    transformers_version='4.26',
    pytorch_version='1.13',
    py_version='py39',
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.xlarge'
)
# predictor.predict({"inputs": "anomaly description here"})
```